# Colab preparation (one pull for all cells below)

Clone on first run, then `git fetch` + `reset --hard origin/main`, mount Google Drive, install missing packages. **Training, task, diagnose, and SVG cells do not run git**—run this cell once per Colab session (and after we push updates).

Prints the repo commit hash. Data: `My Drive/Foundation Model/vendor/pfdelta_task1.1`.


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        subprocess.check_call(["git", "clone", REPO_URL, str(REPO)])
    else:
        subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin"])
        subprocess.check_call(["git", "-C", str(REPO), "reset", "--hard", "origin/main"])
    os.chdir(REPO)
    head = subprocess.check_output(
        ["git", "-C", str(REPO), "log", "-1", "--oneline"], text=True
    ).strip()
    print(f"Repo at {head}", flush=True)
    plot_py = REPO / "plot_task1_1.py"
    if plot_py.is_file() and "display_task31_table" not in plot_py.read_text(encoding="utf-8"):
        raise RuntimeError(
            "plot_task1_1.py is stale after git reset; delete /content/genco-figure4-task11 and re-run prep."
        )
    from google.colab import drive

    drive.mount("/content/drive")

missing = []
for module, package in (
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("matplotlib", "matplotlib"),
    ("pyarrow", "pyarrow"),
    ("huggingface_hub", "huggingface_hub"),
):
    try:
        importlib.import_module(module)
    except ImportError:
        missing.append(package)
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Required packages are already installed.")


# Training V7

This cell trains **TPGT** only (type-pair graph Transformer, independent weights per stage) on that Drive dataset. It writes checkpoints to `Foundation Model/tpgt_task11_v7` and comparison tables to `Foundation Model/results/tpgt_task11_v7` (the folder name follows `RUN_TAG` in the script, so older runs such as `Foundation Model/tpgt_task11` are never overwritten). It does not draw the figure.

Use a **GPU** runtime. Run the prep cell first so you have the latest `train_tpgt_task11.py` from GitHub. A finished seed (100 epochs, matching checkpoint version) is skipped if this cell is run again; obsolete Drive checkpoints (wrong version, old RPE width, or corrupt files) are ignored and training restarts from epoch 0.

Training uses feasible **N** only (same split as GENCO). **100 epochs**, batch **64** (set `BATCH_SIZE` in the training cell; micro-batch matches), loss **0.5** state + **0.5** physics, seeds 0–2. Set **`DATA_FRACTION`** in the training cell (e.g. **0.25**) to use the **first** that fraction of scenarios in each train/valid/test folder (fixed parquet order). Raising it to **0.30** keeps the same first 25% and **adds** the next 5%; cached splits extend instead of rebuilding from scratch. **1.0** is the full dataset.

Optional: set `GENCO_TPGT_COMPILE=1` (or pass `--compile` / `--fast` to `%run`) for `torch.compile`; default training already uses batched type-pair attention.

Pair features are a 5-channel **meshed-network RPE** (full topology: \(\mathrm{Re}\,Y_{ij}\), \(\mathrm{Im}\,Y_{ij}\), adjacency, hop distance, effective resistance), not a spanning tree. During training each RPE channel is zeroed with probability 0.1 per step (never at validation/test). Width/depth (48 hidden, 24 pair/attn, 2 heads, 4 correction stages, 3-layer MLPs) are implementation choices, not the authors' hyperparameters.


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# 1.0 = full PFΔ split; 0.25 ≈ 12150 train / 1350 valid / 500 test (N) cases.
# Batch 64 matches GENCO's step size; drop to 32 if L4 OOMs (dense type-pair attention).
DATA_FRACTION = 1.0
BATCH_SIZE = 64
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(f"DATA_FRACTION={DATA_FRACTION}, BATCH_SIZE={BATCH_SIZE}", flush=True)

%run train_tpgt_task11.py


# Training v8 (deeper + wider, same logic)

Same packing, AC residual, state+physics loss, and RPE as v7 — **no** ignored unknowns / reconstructed outputs. Changes only capacity:

| | v7 | v8 |
| --- | --- | --- |
| `DEPTH` | 4 | **8** |
| `D` | 48 | **96** |
| `D_E` / `D_A` | 24 | **48** |
| `HEADS` | 2 | **4** |
| Drive folder | `tpgt_task11_v7` | `tpgt_task11_v8` |

Run the prep cell first. This does **not** resume v7 weights (new `checkpoint_version`). Prefer `DATA_FRACTION=1.0` for a fair GENCO comparison; drop `BATCH_SIZE` if the L4 OOMs.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# v8: DEPTH=8, D=96, D_E=D_A=48, HEADS=4 (same loss/residual/RPE as v7).
ARCH = "v8"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM; Task 1.1 can go higher
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(f"ARCH={ARCH}, DATA_FRACTION={DATA_FRACTION}, BATCH_SIZE={BATCH_SIZE}", flush=True)

%run train_tpgt_task11.py


# Training v9 (v8 capacity + (P,Q,V,θ,BS) + Zeff Re/Im RPE)

Same packing/residual/loss as v8. **No ground node.**

- **Node in (all types):** (P, Q, V, θ, B_s) = **5**. Unknowns among P/Q/V/θ are **zeroed**.
- **B_s:** parquet bus column `BS` (p.u. **bus shunt device only**). Not |Y_ii|, not Im(sum_j Y_ij) (that still mixes in line charging).
- **Pair features:** **Zeff_Re_norm + Zeff_Im_norm** (complex Laplacian).

| | v8 | v9 |
| --- | --- | --- |
| Capacity | DEPTH=8, D=96, ... | same |
| Node in | specified (2) | **P,Q,V,θ + bus BS (5)** |
| RPE | … | **Zeff Re + Im (2)** |
| Drive folder | 	pgt_task11_v8 | 	pgt_task11_v9 |

Run the prep cell first. Does **not** resume v8 weights (new checkpoint_version).

Changing **`DATA_FRACTION`** (or arch) **purges stale split caches** automatically (`/content/tpgt_task11_v9_cache` + Drive `.../tpgt_task11_v9/cache`) so RPE tensors stay consistent — no manual wipe.


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# v9: v8 capacity + 5 node features (spec + Gs,Bs + |Yii|) for PQ/PV/slack; no ground node.
ARCH = "v9"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM; Task 1.1 can go higher
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(f"ARCH={ARCH}, DATA_FRACTION={DATA_FRACTION}, BATCH_SIZE={BATCH_SIZE}", flush=True)

%run train_tpgt_task11.py


# Training v9 on IEEE 57 (Task 1.1 protocol)

Same **v9** model and Task 1.1 protocol as the IEEE-118 cell above, but **train + test on IEEE 57**:

- Train / valid: feasible **N** only (MIT PFΔ Task 1.1 90/10 split).
- Test: N / N-1 / N-2 feasible + nose (C2I), same reporting as 118.
- Data: builds a gridfm-style parquet pack from `pfdelta/pfdelta` `case57.tar.gz` into `Foundation Model/vendor/pfdelta_task1.1_case57` (first run downloads + converts; later runs reuse).
- Drive folder: `tpgt_task11_case57_v9` (checkpoints, cache, bus error maps) and `results/tpgt_task11_case57_v9`.

Published GENCO / CANOS / GNS-S / PFNet Task 1.1 numbers are **118-only**, so the comparison figure shows **TPGT alone** on 57 (no false overlay).

Run the prep cell first. Set `GENCO_TPGT_CASE=case57` (this cell does). Does **not** share weights or caches with `tpgt_task11_v9`.


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import pull_latest

    print(f"Training from {pull_latest()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v9"
CASE = "case57"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"Task 1.1 ARCH={ARCH} CASE={CASE} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE}",
    flush=True,
)

%run train_tpgt_task11.py


# Training v13 (TPGT-EFLA) on IEEE 57 (Task 1.1 protocol)

**TPGT-EFLA**: v9 capacity (D=96, 4 heads, 8 stages, PQ/PV/Slack, AC residual + state/physics loss) with **electrically factorized linear attention** instead of dense Zeff + GRIT.

- Landmark complex-Laplacian node embeddings `a_i` (16 landmarks → 34-D), no dense `N×N` Zeff in attention.
- Learned operating-state × electrical kernels (32×4=128) and rank-4 separable pair-conditioned values.
- Same Task 1.1 protocol: train/valid feasible **N**; test N / N-1 / N-2 + C2I.
- Data: `Foundation Model/vendor/pfdelta_task1.1_case57` (same pack as the v9 case57 cell).
- Drive folder: `tpgt_task11_case57_v13` (new `checkpoint_version`; does **not** resume v9).

| | v9 | v13 |
|---|---|---|
| Attention | GRIT + dense Zeff RPE | EFLA (factorized, landmark `a_i`) |
| Heads | 4 | 4 |
| η_angle | 1 | 1 |
| Drive folder | `tpgt_task11_case57_v9` | `tpgt_task11_case57_v13` |

Run the prep cell first. Set `GENCO_TPGT_CASE=case57` (this cell does).

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import pull_latest

    print(f"Training from {pull_latest()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v13"
CASE = "case57"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"Task 1.1 ARCH={ARCH} CASE={CASE} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE}",
    flush=True,
)

%run train_tpgt_task11.py


# Training v14 (TPGT-WIRE) on IEEE 57 (Task 1.1 protocol)

**TPGT-WIRE**: v9 capacity (D=96, 4 heads, 8 stages, PQ/PV/Slack, AC residual + state/physics loss) with **electrically weighted WIRE** instead of dense Zeff + GRIT.

- Series-branch susceptance Laplacian \(L_B\) (shunts excluded from the graph; kept in \(Y_{\mathrm{bus}}\) physics).
- Spectral coordinates \(\rho_i\) from the first 8 nonconstant eigenvectors, scaled by \(1/\sqrt{\lambda_k}\).
- Learned per-stage/head frequencies \(\Omega\) rotate Q/K (shared across the 9 type pairs within a head).
- Full softmax attention with same-component mask and no self-attention.
- Same Task 1.1 protocol and case57 data pack as the v9/v13 cells.
- Drive folder: `tpgt_task11_case57_v14` (new `checkpoint_version`; does **not** resume v9/v13).
- Loss / \(\eta_{\mathrm{angle}}\): same as implemented v9 (**0.5/0.5**, **1.0**) for a controlled comparison with existing case57 runs.

| | v9 | v14 |
|---|---|---|
| Attention | GRIT + dense Zeff RPE | WIRE RoPE + full softmax |
| Structural features | Complex \(Z^{\mathrm{eff}}\) | Susceptance-Laplacian \(\rho\) |
| Heads | 4 | 4 |
| Drive folder | `tpgt_task11_case57_v9` | `tpgt_task11_case57_v14` |

Run the prep cell first. Set `GENCO_TPGT_CASE=case57` (this cell does).

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import pull_latest

    print(f"Training from {pull_latest()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v14"
CASE = "case57"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"Task 1.1 ARCH={ARCH} CASE={CASE} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE}",
    flush=True,
)

%run train_tpgt_task11.py


# Training v15 (pairwise electrical rotary) on IEEE 57 (Task 1.1 protocol)

**TPGT v15**: keep v9 capacity (D=96, 4 heads, 8 stages, PQ/PV/Slack, AC residual + 0.5/0.5 state/physics loss) and keep **rotary Q/K**, but replace WIRE’s spectral coordinates with **basis-invariant pairwise electrical features** from the series-susceptance Laplacian \(L_B\).

- Effective electrical distance \(d_{ij}=(e_i-e_j)^\top L_B^\dagger(e_i-e_j)\).
- Pair features \(\mathbf{e}_{ij}=[\log(1+d_{ij}/s_d),\,\mathbb{1}_{(i,j)\in\mathcal{E}_{\mathrm{on}}},\,\log(1+w_{ij}/s_w)]\) with train-fit scales \(s_d,s_w\).
- Pair-dependent angles \(\boldsymbol{\vartheta}_{ij}^{(\ell,h)}=\Omega^{(\ell,h)}\mathbf{e}_{ij}\) with \(\Omega\in\mathbb{R}^{24\times 3}\) per head/stage.
- Score \(s_{ij}=(q_i)^\top R(\boldsymbol{\vartheta}_{ij})k_j/\sqrt{48}\), then full softmax + same-component mask (no self).
- Same Task 1.1 / case57 protocol as the v9 / v13 / v14 cells (controlled comparison).
- Drive folder: `tpgt_task11_case57_v15` (new `checkpoint_version`; does **not** resume v14).

| | v14 | v15 |
|---|---|---|
| Attention | WIRE spectral RoPE + softmax | Pairwise electrical rotary + softmax |
| Structural features | Graph Laplacian eigenvectors \(\rho\) | \(d_{ij}\), direct edge, susceptance weight |
| Heads / depth / D | 4 / 8 / 96 | 4 / 8 / 96 |
| Drive folder | `tpgt_task11_case57_v14` | `tpgt_task11_case57_v15` |

Run the prep cell first. This cell sets `GENCO_TPGT_CASE=case57`.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import pull_latest

    print(f"Training from {pull_latest()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v15"
CASE = "case57"
DATA_FRACTION = 1.0
LEARNING_RATE = 2e-4  # lowered for DEPTH 8–12 stability; resumes from last.pt
BATCH_SIZE = 64  # drop to 32 if OOM (pairwise rotary is denser than WIRE)
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_LR"] = str(LEARNING_RATE)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"Task 1.1 ARCH={ARCH} CASE={CASE} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE} LR={LEARNING_RATE} (resume last.pt if present)",
    flush=True,
)

%run train_tpgt_task11.py


# Training v16 (v9 GRIT + DEPTH=12) on IEEE 57 (Task 1.1 protocol)

Same **v9** architecture (GRIT + Zeff RPE, node features `(P,Q,V,θ,busBS)`, D=96, 4 heads, 0.5/0.5 state/physics loss) with **12 correction stages** instead of 8.

- Controlled comparison vs `tpgt_task11_case57_v9` (DEPTH=8) on the same Task 1.1 / case57 pack.
- Drive folder: `tpgt_task11_case57_v16` (new `checkpoint_version`; does **not** resume v9).

| | v9 | v16 |
|---|---|---|
| Attention / RPE | GRIT + Zeff | GRIT + Zeff |
| DEPTH | 8 | **12** |
| D / heads | 96 / 4 | 96 / 4 |
| Drive folder | `tpgt_task11_case57_v9` | `tpgt_task11_case57_v16` |

Run the prep cell first. This cell sets `GENCO_TPGT_CASE=case57`.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import pull_latest

    print(f"Training from {pull_latest()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v16"
CASE = "case57"
DATA_FRACTION = 1.0
LEARNING_RATE = 2e-4  # lowered for DEPTH 8–12 stability; resumes from last.pt
BATCH_SIZE = 64  # drop to 32 if OOM (12 stages uses more memory)
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_LR"] = str(LEARNING_RATE)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"Task 1.1 ARCH={ARCH} CASE={CASE} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE} LR={LEARNING_RATE} (resume last.pt if present)",
    flush=True,
)

%run train_tpgt_task11.py


# Training v17 (v16 GRIT + Im-only Zeff) on IEEE 57 (Task 1.1 protocol)

Same as **v16** (GRIT, DEPTH=12, D=96, 4 heads, busBS) but **drops Real Zeff** from the RPE. Pair features are **`Zeff_Im_norm` only** (1 channel). Case57 ablation showed `Zeff_Re=0` barely moves PBL.

- Drive folder: `tpgt_task11_case57_v17` (new `checkpoint_version`; does **not** resume v16).
- Cache bump: `tpgt-cache-15` (RPE width 1 vs v16’s 2).

| | v16 | v17 |
|---|---|---|
| Attention | GRIT | GRIT |
| RPE | Zeff Re + Im | **Zeff Im only** |
| DEPTH | 12 | 12 |
| Drive folder | `tpgt_task11_case57_v16` | `tpgt_task11_case57_v17` |

Run the prep cell first. This cell sets `GENCO_TPGT_CASE=case57`.

In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import pull_latest

    print(f"Training from {pull_latest()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v17"
CASE = "case57"
DATA_FRACTION = 1.0
LEARNING_RATE = 2e-4
BATCH_SIZE = 64  # drop to 32 if OOM
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_LR"] = str(LEARNING_RATE)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(
    f"Task 1.1 ARCH={ARCH} CASE={CASE} DATA_FRACTION={DATA_FRACTION} BATCH_SIZE={BATCH_SIZE} LR={LEARNING_RATE} (Im-only Zeff; resume last.pt if present)",
    flush=True,
)

%run train_tpgt_task11.py


# Training v10 (v9 capacity + joint MLP attention on raw RPE)

Same packing, AC residual, state+physics loss, and node features as **v9** — but **Approach C joint MLP attention** instead of GRIT, and **no pair encoder**. Angle damping \(\eta_{\mathrm{angle}}=1\) (same as v9).

- **Capacity:** DEPTH=8, D=96, D_A=48, HEADS=2; score MLP **98→128→64→1**.
- **Node in:** (P, Q, V, θ, busBS) = 5; unknowns zeroed.
- **Raw RPE** \(r_{ij}\): Zeff_Re_norm + Zeff_Im_norm (2 channels) fed **directly** into the score MLP (Identity instead of `pair_encoder`).
- **Type-pair Q/K/V** (projection depends on sender type as well as receiver):
  \[
  q_{i\mid t_j}^{(\ell,h)}=W_Q^{(\ell,h,t_i\leftarrow t_j)}h_i^{(\ell-1)},\quad
  k_{j\mid t_i}^{(\ell,h)}=W_K^{(\ell,h,t_i\leftarrow t_j)}h_j^{(\ell-1)}.
  \]
- **Attention score:** 9 type-pair-specific MLPs × HEADS × DEPTH:
  \(s_{ij}^{(\ell,h)}=\mathrm{MLP}_{t_i\leftarrow t_j}^{(\ell,h)}([q_{i\mid t_j}\Vert k_{j\mid t_i}\Vert r_{ij}])\) with **98→128→64→1**, then softmax over senders.
- **Values:** \(v_{j\mid t_i}^{(\ell,h)}=W_V^{(\ell,h,t_i\leftarrow t_j)}h_j^{(\ell-1)}\) (no GRIT edge-augmented interaction).
- **Drive folder:** `tpgt_task11_v10` (does **not** resume v9 / older MLP widths; new `checkpoint_version`).

| | v9 | v10 |
| --- | --- | --- |
| Capacity / node | DEPTH=8, HEADS=4, (P,Q,V,θ,BS) | DEPTH=8, **HEADS=2**, score MLP **98→128→64→1** |
| \(\eta_{\mathrm{angle}}\) | 1 | **1** |
| RPE path | `pair_encoder` → GRIT | **raw** \(r_{ij}\) → joint MLP |
| Pair compatibility | GRIT multiplicative | **joint MLP** 98→128→64→1 (9×heads×stages) |
| Drive folder | `tpgt_task11_v9` | `tpgt_task11_v10` |

Run the prep cell first. Reporting matches Task 1.1 v9 (checkpoint epochs write bar plots vs published GENCO / CANOS / GNS-S / PFNet, bus error maps, and test PBL tables under the run folder).


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# v10: v9 capacity/features + joint MLP attention (Approach C) instead of GRIT.
ARCH = "v10"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM; Task 1.1 can go higher
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(f"ARCH={ARCH}, DATA_FRACTION={DATA_FRACTION}, BATCH_SIZE={BATCH_SIZE}", flush=True)

%run train_tpgt_task11.py


# Training v11 (Park et al. full GRPE)

Same packing, AC residual, state+physics loss, node features, HEADS=2, DEPTH=8 as **v10**, but attention is **Park GRPE**: shared hop-distance and edge-relation embeddings in both the score and the values.

\[
s_{ij}=\frac{q_i^\top k_j+q_i^\top(P_{ij}^Q+E_{ij}^Q)+k_j^\top(P_{ij}^K+E_{ij}^K)}{\sqrt{D_A}},\quad
o_i=\sum_j\alpha_{ij}(v_j+P_{ij}^V+E_{ij}^V).
\]

- **Capacity:** DEPTH=8, D=96, D_A=48, HEADS=2.
- **Topology:** hop bins `0..8` + overflow; edge ids `{self, in-service branch, non-adjacent}`.
- **Type-pair** \(W_Q,W_K,W_V\) + component masks unchanged (TPGT adaptation).
- **Drive folder:** `tpgt_task11_v11` (new `checkpoint_version`; does not resume v10).
- **\(\eta_{\mathrm{angle}}=1\)**.

| | v10 | v11 |
| --- | --- | --- |
| Score / values | joint MLP; node-only V | **Park GRPE** \(P/E\) in score and values |
| Drive folder | `tpgt_task11_v10` | `tpgt_task11_v11` |

Run the prep cell first. Reporting matches Task 1.1 (bar plots, bus maps, test PBL tables).


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# v11: Park et al. full GRPE (shared hop/edge embeddings + graph-encoded values).
ARCH = "v11"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM; Task 1.1 can go higher
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(f"ARCH={ARCH}, DATA_FRACTION={DATA_FRACTION}, BATCH_SIZE={BATCH_SIZE}", flush=True)

%run train_tpgt_task11.py


# Training v12 (Exphormer sparse graph on v10)

Same packing, AC residual, state+physics loss, node features, HEADS=2, DEPTH=8, and **joint MLP scores** as **v10**, but attention runs on an Exphormer interaction graph: physical branches ∪ random expander (d=4) ∪ one virtual node per energized component.

- **Capacity:** DEPTH=8, D=96, D_A=48, HEADS=2; score MLP 98→128→64→1 on raw Zeff RPE (real–real) / learned virt RPE.
- **Relations:** 15 directed type-pairs (PQ/PV/Slack/U; no Virtual←Virtual).
- **Kernel:** Default kernel is **dense** (−∞) mask (fast on IEEE 118); sparse via GENCO_TPGT_EXPHORMER_DENSE=0.
- **Drive folder:** `tpgt_task11_v12` (new `checkpoint_version`; does not resume v10/v11).
- **η_angle = 1**.

| | v10 | v12 |
| --- | --- | --- |
| Attention support | component-masked dense N×N | phys ∪ expander ∪ virtual |
| Score | joint MLP | same MLP + edge-type bias |
| Drive folder | `tpgt_task11_v10` | `tpgt_task11_v12` |

Run the prep cell first. Reporting matches Task 1.1 (bar plots, bus maps, test PBL tables).


In [ ]:
import os
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")

if on_colab():
    if not Path("/content/drive/MyDrive").is_dir():
        from google.colab import drive

        drive.mount("/content/drive")
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
elif WINDOWS_DATA.is_dir():
    os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

# v12: Exphormer phys∪expander∪virtual graph + v10 joint MLP scores (dense mask default).
ARCH = "v12"
DATA_FRACTION = 1.0
BATCH_SIZE = 64  # drop to 32 if OOM; Task 1.1 can go higher
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
print(f"ARCH={ARCH}, DATA_FRACTION={DATA_FRACTION}, BATCH_SIZE={BATCH_SIZE}", flush=True)

%run train_tpgt_task11.py


# Other PFΔ tasks (IEEE 118)

Each cell below trains TPGT on one PFΔ task with the same model, loss and schedule as Task 1.1. Run the preparation cell once per runtime, then the cell of the task you want (one task per runtime at a time; separate Colab sessions can run different tasks because every task has its own folders). Re-running a cell resumes from that task's last checkpoint.

| Task | Training / validation data | Train scenarios | Drive run folder |
| --- | --- | --- | --- |
| 1.1 | N (cell above) | 48,600 | `tpgt_task11_v7` |
| 1.2 | N + N-1 | 48,600 | `tpgt_task1.2_v7` |
| 1.3 / 2.1 | N + N-1 + N-2 (one dataset, one folder) | 48,600 | `tpgt_task1.3_v7` |
| 2.3 | N + N-1 + N-2, 3x less data | 16,200 | `tpgt_task2.3_v7` |
| 3.1 | Table 4: train 118 (N+N-1+N-2); test 57 / 118 / 500 — **eval-only cell** scores existing v9 weights (no train) | — | `tpgt_task3.1_v9` (results only; weights from `tpgt_task11_v9`) |
| 4.1 | 90% feasible + 10% close-to-infeasible (C2I) | 48,600 | `tpgt_task4.1_v7` |
| 4.2 | 50% feasible + 10% C2I + 40% approaching infeasibility | 48,600 | `tpgt_task4.2_v7` |
| 4.3 | 20% C2I + 80% approaching infeasibility | 48,600 | `tpgt_task4.3_v7` |

Every task is tested on the same six folders as Task 1.1 (N, N-1, N-2 feasible and the three C2I sets).

* **Data**: `My Drive/Foundation Model/vendor/pfdelta_task<id>` (Task 2.1 uses `pfdelta_task1.3`). If it is missing or incomplete, the first run downloads `gridfm/pfdelta_task<id>` from Hugging Face into that folder (about 0.3 to 0.7 GB).
* **`DATA_FRACTION`** at the top of each cell keeps the first fraction of every train/valid/test folder, so the task's mix (N / N-1 / N-2, feasible / C2I / approaching) is preserved. Raising it later extends the cached prefixes.
* **First run** builds split caches (a few minutes; N-2 folders hold thousands of topologies). They are mirrored to `<run folder>/cache` on Drive (up to about 4 GB for tasks 1.3, 4.1 and 4.2) and can be deleted at any time; they are rebuilt from the parquet files.
* **Every checkpoint epoch (3, 6, ...)** writes into the run folder: `seed_<s>/epoch_<e>.pt`, the test-PBL tables and the bar plot `results/figure4_task<id>_tpgt.png/.svg` (TPGT mean ± std over the seeds so far vs. the published GENCO Base, CANOS-PF, GNS-S and PFNet numbers for that task), and IEEE 118 bus error maps in `bus_error_maps/seed_<s>/epoch_<e>/`. Each cell shows the latest bar plot and maps when training stops.
* Batches of 32 cases mix topologies; each topology in a batch gets its own forward pass, so tasks with N-1/N-2 training data take longer per epoch than Task 1.1.


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "1.2"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "1.3"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "2.3"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


### Task 2.3 — **v9d4** (v9 node/RPE, **4** correction stages)

Same as Task 1.1 **v9**: (P,Q,V,θ,busBS) node in + Zeff Re/Im RPE, `η_angle=1`, 50/50 state/physics loss — but **DEPTH=4** (v7 depth). Checkpoints go to `tpgt_task2.3_v9d4` (not `tpgt_task2.3_v7`). Does not load v7 or v9 (8-layer) weights.

**Memory:** Task 2.3 has thousands of N-2 topologies. Training does not globally shuffle those grids (that OOMs). It packs at most 8 same-bus-type grids and 16 cases per step; a chunk of 16+ cases stays one grid. A progress line prints about once a minute (`step k/N`, elapsed, ETA). If a step OOMs, set `GENCO_TPGT_PACK_TOPOLOGIES=4` or `1`. Restart the Colab runtime before training if a prior run OOM’d. Use `BATCH_SIZE=32`, `GENCO_TPGT_EVAL_BATCH=32` (not 64). If it still OOMs, try `BATCH_SIZE=16` and `GENCO_TPGT_MICRO_BATCH=16`.

**Reuse caches:** Keep `tpgt_task2.3_v9d4/cache/` on Drive and the same `ARCH` / `DATA_FRACTION=1.0`. New runtimes mirror those `.pt` files to `/content` (fast load). Only `GENCO_TPGT_REBUILD_CACHE=1` or changing arch/fraction/vendor data forces a rebuild.


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "2.3"
ARCH = "v9d4"
DATA_FRACTION = 1.0
BATCH_SIZE = 32
EVAL_BATCH = 32

REPO = Path("/content/genco-figure4-task11")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{TASK}_{ARCH}"
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_MICRO_BATCH"] = str(BATCH_SIZE)
os.environ["GENCO_TPGT_EVAL_BATCH"] = str(EVAL_BATCH)
print(
    f"Task {TASK} ARCH={ARCH}: DATA_FRACTION={DATA_FRACTION}, "
    f"BATCH_SIZE={BATCH_SIZE}, EVAL_BATCH={EVAL_BATCH}, run folder {RUN}",
    flush=True,
)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


### Task 3.1 — **v9** (8 layers, eval only)

GENCO **Table 4** protocol: dataset `gridfm/pfdelta_task3.1` on Drive (`vendor/pfdelta_task3.1`), same split-cache rules as other tasks (`reconcile_split_caches`, train RPE scales, test caches under `tpgt_task3.1_<ARCH>/cache`). This cell **does not train** — it loads **Task 1.1 v9** weights from `tpgt_task11_v9` and scores **IEEE 57 / 118 / GOC 500** (cross-grid eval copies `s_R,s_X` from the weight run, not from Task 3.1 train parquet).

Results: `Foundation Model/tpgt_task3.1_v9/results/` (CSV, summary, three-panel bar, **`table4_task3.1_tpgt.html` / `.md`** — paper Table 4 with NR row, TPGT row, bold best learning score per column). To reproduce paper training (118 N+N-1+N-2 then Table 4 test), run `%run train_tpgt_task11.py --task 3.1` instead of `--eval-only`.

**First run:** builds 57- and 500-bus **test** split caches (several minutes); each `.pt` is mirrored to `tpgt_task3.1_<ARCH>/cache/` on **Drive** and reloaded on the next session. Scores are checkpointed per grid in `results/task3_1_tpgt_eval_partial.json` so a 500-bus OOM does not redo finished grids. Use `EVAL_BATCH=16` (500-bus is auto-capped).


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "3.1"
ARCH = "v9"
DATA_FRACTION = 1.0
EVAL_BATCH = 16  # 500-bus is auto-capped; never use 64 on Task 3.1

REPO = Path("/content/genco-figure4-task11")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Eval from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{TASK}_{ARCH}"
TRAINED = DRIVE / "Foundation Model" / f"tpgt_task11_{ARCH}"

os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_TASK"] = TASK
os.environ["GENCO_PFDELTA_TASK31"] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
os.environ["GENCO_TPGT_CHECKPOINT_ROOT"] = str(TRAINED)
os.environ["GENCO_TPGT_EVAL_BATCH"] = str(EVAL_BATCH)
print(
    f"Task {TASK} eval-only ARCH={ARCH}: DATA_FRACTION={DATA_FRACTION}, "
    f"weights {TRAINED}, results cache run {RUN}, EVAL_BATCH={EVAL_BATCH}",
    flush=True,
)

%run train_tpgt_task11.py --task {TASK} --eval-only

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
else:
    print(f"Figure not found yet: {bar}", flush=True)

table_html = RUN / "results" / "table4_task3.1_tpgt.html"
if table_html.is_file():
    from IPython.display import HTML, display

    display(HTML(table_html.read_text(encoding="utf-8")))


### GENCO §5.4 — **Figure 8** runtime + **Table 6** (TPGT vs published)

In-memory PF protocol (gridfm-graphkit `genco-paper-repro`): preload scenarios, warmup excluded, amortized **ms/instance** at best batch (IEEE from 64, GOC from 16). Published curves come from `data/pf_runtime_published.json` (refresh on Colab with `scripts/build_pf_runtime_published.py` when online).

**Hardware gate:** paper timings use an exclusive **NVIDIA H100 SXM5 80GB**. This cell **stops before TPGT timing** on any other GPU (T4/A100/L4 are not comparable). Weights: Task **1.1 v9**; scenarios: Task **3.1** test **N** splits for IEEE **57 / 118 / GOC 500** (same caches as Table 4 eval).

Outputs under `Foundation Model/tpgt_task3.1_v9/results/`: `figure8_runtime_pf.png`, `table6_runtime_grid*.html`, `runtime_tpgt_bench.json`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ARCH = "v9"
REBENCH = True  # set False to reuse results/runtime_tpgt_bench.json

REPO = Path("/content/genco-figure4-task11")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    ensure_training_cwd()
    print(f"§5.4 runtime from {pull_latest()}", flush=True)
    if not (REPO / "plot_runtime_fig8.py").is_file():
        raise RuntimeError("plot_runtime_fig8.py missing; git pull / prep again.")
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "plot_runtime_fig8.py").is_file():
        os.chdir(here / "genco")
        REPO = here / "genco"
    elif not (here / "plot_runtime_fig8.py").is_file():
        raise FileNotFoundError("plot_runtime_fig8.py is not in this folder.")
    sys.path.insert(0, str(REPO))
    DRIVE = Path(r"K:\My Drive")

TRAINED = DRIVE / "Foundation Model" / f"tpgt_task11_{ARCH}"
RUN = DRIVE / "Foundation Model" / f"tpgt_task3.1_{ARCH}"
DATA = DRIVE / "Foundation Model" / "vendor" / "pfdelta_task3.1"
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_TASK"] = "3.1"
os.environ["GENCO_PFDELTA_TASK31"] = str(DATA)
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"

pub = REPO / "data" / "pf_runtime_published.json"
if not pub.is_file():
    subprocess.check_call([sys.executable, str(REPO / "scripts" / "build_pf_runtime_published.py")])

import importlib
import plot_runtime_fig8
importlib.reload(plot_runtime_fig8)
from plot_runtime_fig8 import display_runtime_fig8, run_notebook_section54

results = run_notebook_section54(
    data_root=DATA,
    weight_run=TRAINED,
    task_run=RUN,
    arch=ARCH,
    rebench=REBENCH,
    published_json=pub,
)
display_runtime_fig8(results)


### §5.4-style runtime on **L4** (matched GENCO vs several TPGTs)

Same in-memory PF protocol as the paper cell: preload, warmup excluded, amortized **ms/instance**.

Times **multiple TPGT checkpoints** on one GPU and overlays them against GENCO Tiny/Small/Base on **this L4**.

- **Matched GENCO** (figure): one batch per grid = max TPGT-feasible batch.
- **GENCO batch sweep** (`GENCO_BATCH_SWEEP=True`): Tiny/Small/Base at **every** batch size (same sweep as TPGT). Writes `runtime_genco_batch_sweep_l4_case118.csv`.

**`GRIDS`:** default `("case118",)` for IEEE-118 only. Set `("case57", "case118", "case500")` for all three.

**`RUNS`:** `{label, arch, folder}` under Drive `Foundation Model/` (needs `seed_0/`).

For **GENCO-only** (skip re-timing TPGTs), use the next cell.
\n

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# --- edit these ---
# Grids: IEEE-118 only by default. All three: ("case57", "case118", "case500")
GRIDS = ("case118",)

# Overlay these checkpoints vs matched GENCO (folder under Foundation Model/)
RUNS = [
    {"label": "v9", "arch": "v9", "folder": "tpgt_task11_v9"},
    {"label": "v11", "arch": "v11", "folder": "tpgt_task11_v11"},
    {"label": "v12", "arch": "v12", "folder": "tpgt_task11_v12"},
    {"label": "v9d4 (task2.3)", "arch": "v9d4", "folder": "tpgt_task2.3_v9d4"},
    {"label": "v13 (case57)", "arch": "v13", "folder": "tpgt_task11_case57_v13"},
]

GPU_NEEDLE = "L4"
REBENCH = True  # force retime each TPGT
MATCH_GENCO = True
REBENCH_GENCO = True  # force retime matched GENCO + GENCO batch sweep
GENCO_BATCH_SWEEP = True  # Tiny/Small/Base at every batch size (not only matched)
# ------------------

REPO = Path("/content/genco-figure4-task11")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    ensure_training_cwd()
    print(f"L4 multi-TPGT §5.4 from {pull_latest()}", flush=True)
    if not (REPO / "plot_runtime_fig8.py").is_file():
        raise RuntimeError("plot_runtime_fig8.py missing; git pull / prep again.")
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "plot_runtime_fig8.py").is_file():
        os.chdir(here / "genco")
        REPO = here / "genco"
    elif not (here / "plot_runtime_fig8.py").is_file():
        raise FileNotFoundError("plot_runtime_fig8.py is not in this folder.")
    sys.path.insert(0, str(REPO))
    DRIVE = Path(r"K:\My Drive")

fm = DRIVE / "Foundation Model"
RUN = fm / "tpgt_task3.1_multi_l4"
DATA = fm / "vendor" / "pfdelta_task3.1"

missing = []
for entry in RUNS:
    ckpt = fm / entry["folder"] / "seed_0"
    if not ckpt.is_dir():
        missing.append(str(fm / entry["folder"]))
if missing:
    raise FileNotFoundError(
        "Missing seed_0/ for:\n  - " + "\n  - ".join(missing)
    )

os.environ["GENCO_TPGT_TASK"] = "3.1"
os.environ["GENCO_PFDELTA_TASK31"] = str(DATA)
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"
os.environ.setdefault(
    "GENCO_RUNTIME_DATA",
    str(fm / "vendor" / "genco_pf_runtime"),
)

pub = REPO / "data" / "pf_runtime_published.json"
if not pub.is_file():
    subprocess.check_call([sys.executable, str(REPO / "scripts" / "build_pf_runtime_published.py")])

import importlib
import plot_runtime_fig8

importlib.reload(plot_runtime_fig8)
import bench_genco_matched

importlib.reload(bench_genco_matched)
print(f"bench_genco_matched from {bench_genco_matched.__file__}", flush=True)
from plot_runtime_fig8 import (
    display_multi_runtime_fig8,
    normalize_bench_grids,
    run_notebook_multi_tpgt_section54,
)

selected = normalize_bench_grids(GRIDS)
grid_tag = "_".join(selected) if selected != plot_runtime_fig8.TPGT_BENCH_GRIDS else "all"
suffix = "" if GPU_NEEDLE.upper() == "H100" else f"_{GPU_NEEDLE.lower()}"
head_name = f"figure8_genco_vs_multi_tpgt{suffix}_{grid_tag}.png"
csv_name = f"runtime_multi_tpgt{suffix}_{grid_tag}.csv"

print(
    f"Timing {len(RUNS)} TPGTs on {selected} vs GENCO (match_genco={MATCH_GENCO})",
    flush=True,
)
results = run_notebook_multi_tpgt_section54(
    data_root=DATA,
    fm_root=fm,
    runs=RUNS,
    task_run=RUN,
    grids=selected,
    rebench=REBENCH,
    published_json=pub,
    gpu_needle=GPU_NEEDLE,
    match_genco=MATCH_GENCO,
    rebench_genco=REBENCH_GENCO,
    genco_batch_sweep=GENCO_BATCH_SWEEP,
)
display_multi_runtime_fig8(
    results,
    head_to_head_name=head_name,
    csv_name=csv_name,
)
sweep_csv = results / f"runtime_genco_batch_sweep{suffix}_{grid_tag}.csv"
if sweep_csv.is_file():
    import pandas as pd
    from IPython.display import display

    print(f"GENCO batch sweep: {sweep_csv}", flush=True)
    display(pd.read_csv(sweep_csv))


### L4 GENCO-only batch sweep (every batch size)

Times **GENCO Tiny / Small / Base** on the same in-memory protocol at **every** batch size (1… up / OOM), without re-timing TPGTs.

**`GRIDS`:** default `("case118",)`. Use `("case57", "case118", "case500")` for all three.

**Outputs** under `tpgt_task3.1_multi_l4/results/`:
- `runtime_genco_batch_sweep_l4_case118.csv`
- `runtime_genco_batch_sweep_l4_case118.json`


In [ ]:
import os
import sys
from pathlib import Path

# --- edit these ---
GRIDS = ("case118",)  # or ("case57", "case118", "case500")
GPU_NEEDLE = "L4"
REBENCH = True
# ------------------

REPO = Path("/content/genco-figure4-task11")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    ensure_training_cwd()
    print(f"GENCO batch sweep from {pull_latest()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "plot_runtime_fig8.py").is_file():
        os.chdir(here / "genco")
        REPO = here / "genco"
    elif not (here / "plot_runtime_fig8.py").is_file():
        raise FileNotFoundError("plot_runtime_fig8.py is not in this folder.")
    sys.path.insert(0, str(REPO))
    DRIVE = Path(r"K:\My Drive")

fm = DRIVE / "Foundation Model"
RUN = fm / "tpgt_task3.1_multi_l4"
os.environ.setdefault(
    "GENCO_RUNTIME_DATA",
    str(fm / "vendor" / "genco_pf_runtime"),
)

import importlib

import bench_genco_matched
import plot_runtime_fig8

importlib.reload(bench_genco_matched)
importlib.reload(plot_runtime_fig8)
from plot_runtime_fig8 import normalize_bench_grids, run_genco_batch_sweep_section54

selected = normalize_bench_grids(GRIDS)
grid_tag = "_".join(selected) if selected != plot_runtime_fig8.TPGT_BENCH_GRIDS else "all"
suffix = "" if GPU_NEEDLE.upper() == "H100" else f"_{GPU_NEEDLE.lower()}"

results = run_genco_batch_sweep_section54(
    task_run=RUN,
    grids=selected,
    rebench=REBENCH,
    gpu_needle=GPU_NEEDLE,
)
csv_path = results / f"runtime_genco_batch_sweep{suffix}_{grid_tag}.csv"
print(csv_path, flush=True)
if csv_path.is_file():
    import pandas as pd
    from IPython.display import display

    display(pd.read_csv(csv_path))


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "4.1"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "4.2"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


In [ ]:
import os
from pathlib import Path

from IPython.display import Image, display

TASK = "4.3"
DATA_FRACTION = 1.0

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    import sys

    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, git_head, mount_drive_if_needed

    ensure_training_cwd()
    print(f"Training from {git_head()}", flush=True)
    DRIVE = mount_drive_if_needed()
else:
    here = Path.cwd()
    if (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    elif not (here / "train_tpgt_task11.py").is_file():
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    DRIVE = Path(r"K:\My Drive")

DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN = DRIVE / "Foundation Model" / f"tpgt_task{DATA_ID}_v7"
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = str(
    DRIVE / "Foundation Model" / "vendor" / f"pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = str(DATA_FRACTION)
print(f"Task {TASK}: DATA_FRACTION={DATA_FRACTION}, run folder {RUN}", flush=True)

%run train_tpgt_task11.py --task {TASK}

bar = RUN / "results" / f"figure4_task{DATA_ID}_tpgt.png"
if bar.is_file():
    display(Image(filename=str(bar)))
maps = sorted(
    RUN.glob("bus_error_maps/seed_*/epoch_*"),
    key=lambda path: (int(path.name.split("_")[1]), path.parent.name),
)
if maps:
    print(f"Latest bus error maps: {maps[-1]}", flush=True)
    for png in sorted(maps[-1].glob("*.png")):
        display(Image(filename=str(png)))


# Diagnostics and bus error maps

Run these after (or between) training sessions; they cannot run while the training cell is running. Both use the latest checkpoint of the run folder of `TASK` (set at the top of each cell; `"1.1"` is `Foundation Model/tpgt_task11_v7`, other tasks `Foundation Model/tpgt_task<id>_v7`) and the split caches the training cell built.

**Diagnose** prints: oracle / flat-start / per-stage PBL, state RMSE by bus type, RPE channel zero-ablation, RPE and residual correctness audits, and the N-1 / N-2 contingency report split into line vs generator outages.

**case57 v15 ablations** (next cell): pair-elec zero-ablation + N−1/N−2 contingency report on `tpgt_task11_case57_v15`.

**case57 v16 GRIT ablations**: post-training component removals (`W_Eb`, `W_Ev`, Re Zeff, `W_Ew`, heads, stage cutoffs) on `tpgt_task11_case57_v16` — no retrain.

**case57 v16 attention sparsity (Step 2)**: bus×bus attention heatmaps, pairs@90/95/99 out of n² (attention / message-contribution mass), optimistic top-k renormalized PBL — no retrain.

**case57 v16 Experiment B (attn rank)**: SVD energy E(r) of attention / message matrices at r=4/8/16/32 + optimistic low-rank PBL — no retrain.

**Bus error maps** draw per-bus PBL on the IEEE 118 one-line layout (validation N, worst generator outage, worst and median line outage), plus a hop-distance table, and display them below.


In [ ]:
import os
import sys
from pathlib import Path

TASK = "1.1"

REPO = Path("/content/genco-figure4-task11")
if not REPO.is_dir():
    raise RuntimeError("Run the Colab preparation cell first.")
sys.path.insert(0, str(REPO))
from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed

ensure_training_cwd()
mount_drive_if_needed()
DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
os.environ["GENCO_PFDELTA_TASK" + DATA_ID.replace(".", "")] = (
    f"/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task{DATA_ID}"
)
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"

%run train_tpgt_task11.py --diagnose --task {TASK} --seed 0 --cases 32


# Post-training ablations — `tpgt_task11_case57_v15`

Loads the trained **v15** IEEE-57 checkpoint (`Foundation Model/tpgt_task11_case57_v15/seed_0/last.pt`) and runs:

- Validation forward diagnostics + **pair-elec** sample ablation (`elec_dist`, `edge_on`, `edge_weight`)
- Test **N−1 / N−2** contingency class report
- **Full-split N / N−1 / N−2 PBL** for baseline and each pair-elec channel zeroed (same format as the GRIT ablation table)

No retraining. Run the prep cell first. Writes `seed_0/pair_channel_split_ablations.json` on Drive.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    print(f"Ablations from {pull_latest()}", flush=True)
    ensure_training_cwd()
    mount_drive_if_needed()
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    if WINDOWS_DATA.is_dir():
        os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v15"
CASE = "case57"
SEED = 0
CASES = 32  # validation forward sample size
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"
os.environ["GENCO_TPGT_DEBUG"] = "1"
os.environ["GENCO_TPGT_RPE_ABLATION"] = "1"
print(
    f"Diagnose+ablate ARCH={ARCH} CASE={CASE} seed={SEED} "
    f"(Drive: tpgt_task11_{CASE}_{ARCH})",
    flush=True,
)

%run train_tpgt_task11.py --diagnose --task 1.1 --seed {SEED} --cases {CASES}


# Post-training GRIT component ablations — `tpgt_task11_case57_v16`

**Step 1 (already in v16 code):** math-preserving speedups — `W_Ev` aggregate-then-project, cached static `W_Ew e` / `W_Eb e`, persisted `pair_embed`. Same GRIT equations.

Uses the trained **v16** checkpoint (no retrain). Scores feasible **N / N−1 / N−2** PBL after removing one GRIT piece at a time (matmul skipped, not multiplied by zero):

| Priority | Ablation | What is removed |
|---|---|---|
| 1 | `W_Eb=0` | Additive electrical term |
| 2 | `W_Ev=0` | Pair-conditioned value correction |
| 3 | `Zeff_Re=0` / `Zeff_Im=0` | Each Zeff RPE channel (full N/N−1/N−2) |
| 4 | `W_Ew=0` | Multiplicative node–electrical term |
| 5 | `head_h=0` | Each attention head’s output |
| 6 | `stage_cutoff_k` | PBL after stages 8…11 (early exit) |

Writes on Drive under `tpgt_task11_case57_v16/seed_0/`:
`grit_component_ablations.json` and `grit_component_ablations.csv` (session-safe).

Run prep first; needs a finished `tpgt_task11_case57_v16` run.

**Next cells:** Step 2 attention sparsity; then Experiment B (attention SVD rank / E(r) + low-rank PBL).

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    print(f"GRIT ablations from {pull_latest()}", flush=True)
    ensure_training_cwd()
    mount_drive_if_needed()
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    if WINDOWS_DATA.is_dir():
        os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v16"
CASE = "case57"
SEED = 0
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"
print(
    f"GRIT component ablate ARCH={ARCH} CASE={CASE} seed={SEED} "
    f"(Drive: tpgt_task11_{CASE}_{ARCH})",
    flush=True,
)

%run train_tpgt_task11.py --ablate-grit --task 1.1 --seed {SEED}


# Step 2 — Attention sparsity diagnostic — `tpgt_task11_case57_v16`

Optimistic test on the **trained v16** checkpoint (no retrain): keep the GRIT equations, only ask whether all sender buses are needed.

**Outputs are written to Google Drive** (survives Colab runtime loss):
`MyDrive/Foundation Model/tpgt_task11_case57_v16/seed_0/attn_sparsity/`

- **2D attention grids** `heatmap_{N|N-1|N-2}_all_stages_mean.png` (and per-stage / per-head)
- **2D message grids** `heatmap_{N|N-1|N-2}_all_stages_message_mean.png` (α‖v‖, per-stage too)
- **Mass curves** — senders covering 90/95/99% of attention
- **CSV / JSON** `attention_mass_by_stage.csv` — `pairs@90/95/99` out of `n²` (approx. mean senders/row × n), plus `frac@*`
- **JSON** `attn_sparsity_summary.json`
- **`MANIFEST.txt`** — file list + sizes after the run
- **`attn_sparsity_artifacts.zip`** — all PNGs/CSV/JSON in one Drive file (most reliable download)
- **Optimistic top-k PBL** in the summary JSON

Artifacts are staged on local SSD then copied to Drive (Colab Drive FUSE often drops direct PNG writes). Set `CASES` for sample size. Run prep first.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    print(f"Attention sparsity from {pull_latest()}", flush=True)
    ensure_training_cwd()
    mount_drive_if_needed()
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    if WINDOWS_DATA.is_dir():
        os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v16"
CASE = "case57"
SEED = 0
CASES = 16  # scenarios per topology sample (N / N-1 / N-2)
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"
print(
    f"Attn sparsity ARCH={ARCH} CASE={CASE} seed={SEED} cases={CASES} "
    f"(Drive: tpgt_task11_{CASE}_{ARCH}/seed_{SEED}/attn_sparsity)",
    flush=True,
)

import importlib

import attn_sparsity_diag

importlib.reload(attn_sparsity_diag)

%run train_tpgt_task11.py --attn-sparsity --task 1.1 --seed {SEED} --cases {CASES}

importlib.reload(attn_sparsity_diag)
from attn_sparsity_diag import display_attn_sparsity

fm = (
    Path("/content/drive/MyDrive/Foundation Model")
    if on_colab()
    else Path(r"K:\My Drive\Foundation Model")
)
out = fm / f"tpgt_task11_{CASE}_{ARCH}" / f"seed_{SEED}" / "attn_sparsity"
if not out.is_dir():
    raise FileNotFoundError(f"Expected Drive outputs at {out}")
files = sorted(p for p in out.rglob("*") if p.is_file())
print(f"Drive artifacts ({len(files)}) under {out}:", flush=True)
for p in files:
    print(f"  {p.relative_to(out)} ({p.stat().st_size} bytes)", flush=True)
display_attn_sparsity(out)

# Experiment B — Attention numerical rank — `tpgt_task11_case57_v16`

Diagnostic on the **trained v16** checkpoint (no retrain): SVD of each stage/head attention matrix \(A^{(\ell,h)}=U\Sigma V^\top\) and of pair-conditioned message magnitudes \(\alpha\|v\|\).

**Question:** do low ranks preserve energy / predictions?

- Cumulative energy \(E(r)=\sum_{k\le r}\sigma_k^2 / \sum_k\sigma_k^2\) at **r = 4, 8, 16, 32**
- `rank@90/95/99` — smallest r covering that energy
- Optimistic **low-rank PBL**: truncate softmax attention to rank r, clamp≥0, row-renormalize (N / N−1 / N−2)

**Outputs (Drive):** `MyDrive/Foundation Model/tpgt_task11_case57_v16/seed_0/attn_rank/`

- Energy curves `energy_curve_{N|N-1|N-2}_stage*_attention.png` (+ `_message.png`)
- CSV `attention_rank_energy_by_stage.csv`, `attention_rank_pbl.csv`
- JSON / zip `attn_rank_summary.json`, `attn_rank_artifacts.zip`

This does **not** implement a fast low-rank GRIT kernel — SVD here is diagnostic only. Run prep first.


In [ ]:
import os
import sys
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
COLAB_DATA = "/content/drive/MyDrive/Foundation Model/vendor/pfdelta_task1.1_case57"
WINDOWS_DATA = Path(r"K:\My Drive\Foundation Model\vendor\pfdelta_task1.1_case57")


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


if on_colab():
    if not REPO.is_dir():
        raise RuntimeError("Run the Colab preparation cell first.")
    sys.path.insert(0, str(REPO))
    from colab_bootstrap import ensure_training_cwd, mount_drive_if_needed, pull_latest

    print(f"Attention rank from {pull_latest()}", flush=True)
    ensure_training_cwd()
    mount_drive_if_needed()
    os.environ["GENCO_PFDELTA_TASK11"] = COLAB_DATA
else:
    here = Path.cwd()
    if (here / "train_tpgt_task11.py").is_file():
        pass
    elif (here / "genco" / "train_tpgt_task11.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("train_tpgt_task11.py is not in this folder.")
    if WINDOWS_DATA.is_dir():
        os.environ["GENCO_PFDELTA_TASK11"] = str(WINDOWS_DATA)

ARCH = "v16"
CASE = "case57"
SEED = 0
CASES = 16  # scenarios per topology sample (N / N-1 / N-2)
os.environ["GENCO_TPGT_ARCH"] = ARCH
os.environ["GENCO_TPGT_CASE"] = CASE
os.environ["GENCO_TPGT_DATA_FRACTION"] = "1.0"
print(
    f"Attn rank ARCH={ARCH} CASE={CASE} seed={SEED} cases={CASES} "
    f"(Drive: tpgt_task11_{CASE}_{ARCH}/seed_{SEED}/attn_rank)",
    flush=True,
)

import importlib

import attn_rank_diag

importlib.reload(attn_rank_diag)

%run train_tpgt_task11.py --attn-rank --task 1.1 --seed {SEED} --cases {CASES}

importlib.reload(attn_rank_diag)
from attn_rank_diag import display_attn_rank

fm = (
    Path("/content/drive/MyDrive/Foundation Model")
    if on_colab()
    else Path(r"K:\My Drive\Foundation Model")
)
out = fm / f"tpgt_task11_{CASE}_{ARCH}" / f"seed_{SEED}" / "attn_rank"
if not out.is_dir():
    raise FileNotFoundError(f"Expected Drive outputs at {out}")
files = sorted(p for p in out.rglob("*") if p.is_file())
print(f"Drive artifacts ({len(files)}) under {out}:", flush=True)
for p in files:
    print(f"  {p.relative_to(out)} ({p.stat().st_size} bytes)", flush=True)
display_attn_rank(out)


# Bus error maps — IEEE 118 (`tpgt_task11_v7`)

Separate from the case57 v16 Step 2 / Experiment B cells above.

Loads the Task **1.1 v7** checkpoint and draws per-bus PBL on the IEEE 118 one-line layout (validation N, worst generator outage, worst/median line outage), plus a hop-distance table. Writes under Drive `tpgt_task11_v7/bus_error_maps/seed_0/`.

Change `TASK` / `RUN_NAME` if you want maps for another run. Run Colab prep first.


In [ ]:
import sys
from pathlib import Path

from IPython.display import Image, display

REPO = Path("/content/genco-figure4-task11")
if not REPO.is_dir():
    raise RuntimeError("Run the Colab preparation cell first.")
sys.path.insert(0, str(REPO))
from colab_bootstrap import ensure_training_cwd

ensure_training_cwd()

TASK = "1.1"
DATA_ID = "1.3" if TASK in ("1.3", "2.1") else TASK
RUN_NAME = "tpgt_task11_v7" if DATA_ID == "1.1" else f"tpgt_task{DATA_ID}_v7"
RUN = Path("/content/drive/MyDrive/Foundation Model") / RUN_NAME
SEED = 0
CHECKPOINT = RUN / f"seed_{SEED}" / "last.pt"
CACHE = Path(f"/content/{RUN_NAME}_cache")
if not (CACHE / "test_n-1_feasible.pt").is_file():
    CACHE = RUN / "cache"
VAL_CACHE = next(
    name
    for name in ("valid_n_feasible.pt", "valid_n_around_nose.pt", "valid_n_nose.pt")
    if (CACHE / name).is_file()
)
OUT = RUN / "bus_error_maps" / f"seed_{SEED}"

%run plot_bus_error_map.py --checkpoint "{CHECKPOINT}" --cache-dir "{CACHE}" --val-cache "{VAL_CACHE}" --out-dir "{OUT}"

for name in (
    "val_N_mean_pbl.png",
    "val_N_dP_dQ.png",
    "test_N1_gen_outage_worst.png",
    "test_N1_line_outage_worst.png",
    "test_N1_line_outage_median.png",
):
    if (OUT / name).is_file():
        display(Image(filename=str(OUT / name)))


# SVG figure

This cell draws the published Task 1.1 SVG (GENCO Base, CANOS-PF, GNS-S, PFNet) from the published table. It does not train.


In [ ]:
import os
import subprocess
from pathlib import Path

REPO = Path("/content/genco-figure4-task11")
REPO_URL = "https://github.com/alitasavori/genco-figure4-task11.git"


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


here = Path.cwd()
if not (here / "plot_task1_1.py").is_file():
    if on_colab():
        if not REPO.is_dir():
            raise RuntimeError("Run the Colab preparation cell first.")
        os.chdir(REPO)
    elif (here / "genco" / "plot_task1_1.py").is_file():
        os.chdir(here / "genco")
    else:
        raise FileNotFoundError("plot_task1_1.py is not in this folder.")

%run plot_task1_1.py --published
